# RetailSignal — exploratory analysis
A guided review of the published real-data marts. Run after installing the project. This notebook does not require the raw workbook.

Source: Chen (2015), UCI Online Retail, CC BY 4.0, DOI 10.24432/C5BW33. All amounts are GBP.

In [ ]:
from pathlib import Path
import pandas as pd
import plotly.express as px
ROOT = Path.cwd() if (Path.cwd() / "data/exports").exists() else Path.cwd().parent
DATA = ROOT / "data/exports"
daily = pd.read_csv(DATA / "mart_daily_country.csv")
monthly = pd.read_csv(DATA / "mart_monthly.csv")
daily.head()

## 1. Does the ledger reconcile?
Anonymous transactions remain in revenue. Credits are independent ledger events; do not call them a matched return rate.

In [ ]:
totals = daily[["gross_revenue", "credits", "net_revenue"]].sum()
assert abs(totals.gross_revenue - totals.credits - totals.net_revenue) < .01
totals

## 2. What share lacks customer identity?
Explain why financial and customer-modeling populations differ.

In [ ]:
anonymous_share = daily.anonymous_gross.sum() / daily.gross_revenue.sum()
print(f"Anonymous share of gross revenue: {anonymous_share:.1%}")
markets = daily.groupby("country", as_index=False).net_revenue.sum().sort_values("net_revenue", ascending=False)
markets.head(10)

## 3. Is apparent growth a partial-period artifact?
December 2011 is incomplete. Compare complete months before making a growth claim.

In [ ]:
complete = monthly.loc[~monthly.partial_month].copy()
complete["net_growth"] = complete.net_revenue.pct_change()
px.bar(complete, x="month", y="net_revenue", title="Net revenue — complete calendar months")

## 4. How does retention vary by cohort?
Blank cells are unobserved; zeroes are observed inactivity. These are first-observed cohorts, not verified acquisition dates.

In [ ]:
cohorts = pd.read_csv(DATA / "mart_cohorts.csv")
matrix = cohorts.pivot(index="cohort", columns="month_index", values="retention")
px.imshow(matrix, text_auto=".0%", zmin=0, zmax=1, aspect="auto", color_continuous_scale="Teal")

## 5. Does the model beat a simple baseline?
Compare only rows with the same split. Explain average precision, prevalence, calibration, and why lift is not causal uplift.

In [ ]:
metrics = pd.read_csv(DATA / "model_metrics.csv")
metrics.loc[metrics.split == "test", ["model", "average_precision", "brier_score", "precision_at_20pct", "capture_at_20pct"]]

## 6. Translate findings into decisions
Read `reports/executive_brief.md`. Propose one credit audit, one identity-coverage improvement, and one randomized campaign experiment. State what evidence would change each recommendation.